# 08 — Topic network del Guardian (co-occorrenza dei tag)

Stesso grafo del notebook `02_topicNetworkAndKeywordEvolution.ipynb`, costruito sugli articoli del Guardian per poterlo confrontare con quello del NYT: nodi = tag keyword assegnati dalla redazione, archi = numero di articoli in cui due tag compaiono insieme.

**Input**: `data/processed/guardian/articles_ice_relevant.parquet`, solo gli articoli con `is_ice_relevant` (filtro di rilevanza di `05_guardian.ipynb` §5). Nomi e tipi dei tag letti dal raw `data/raw/guardian/search/**/*.json`.

**Cosa è identico al NYT**: `build_cooccurrence_graph`, `prune_graph`, `threshold_table`, periodi, grafo dinamico, soglia `MIN_EDGE_WEIGHT = 5`, attributi dei nodi (`count`, `avg_rank`, `category`).

**Cosa cambia per forza** (i due giornali hanno tassonomie diverse):

| | NYT | Guardian |
|---|---|---|
| Keyword | `keywords[].value` | tag keyword (id `sezione/slug`), etichetta = `webTitle` |
| Nodo ICE escluso | `immigration and customs enforcement (us)` | `us-news/ice-us-immigration-and-customs-enforcement` |
| Categoria | `keywords[].name` | `keywordType` del tag, rimappato sulle categorie NYT |
| Rank | `keywords[].rank` | posizione del tag nella lista dell'articolo (1 = primo) |

**Output** in `data/processed/guardian/`:

- `ice_topic_network.gexf` (corpus intero), `ice_topic_network_pruned.gexf` (diluito)
- `ice_topic_network_{periodo}.gexf`, `ice_topic_network_dynamic.gexf`
- `topic_network_period_stats.csv`, `topic_network_nyt_vs_guardian.csv`
- `plots/edge_weight_hist.png`

## 0. Setup

In [ ]:
import json
import logging
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import Image, display

%matplotlib inline

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = BASE_DIR / "data" / "raw" / "guardian" / "search"
PROCESSED_DIR = BASE_DIR / "data" / "processed" / "guardian"
PLOTS_DIR = PROCESSED_DIR / "plots"
NYT_DIR = BASE_DIR / "data" / "processed" / "nyt"

IN_RELEVANT = PROCESSED_DIR / "articles_ice_relevant.parquet"
IN_NYT_GEXF = NYT_DIR / "ice_topic_network.gexf"
IN_NYT_PRUNED_GEXF = NYT_DIR / "ice_topic_network_pruned.gexf"
OUT_GEXF = PROCESSED_DIR / "ice_topic_network.gexf"
OUT_PRUNED_GEXF = PROCESSED_DIR / "ice_topic_network_pruned.gexf"
OUT_DYNAMIC_GEXF = PROCESSED_DIR / "ice_topic_network_dynamic.gexf"
OUT_PERIOD_STATS = PROCESSED_DIR / "topic_network_period_stats.csv"
OUT_COMPARISON = PROCESSED_DIR / "topic_network_nyt_vs_guardian.csv"
OUT_WEIGHT_HIST_PNG = PLOTS_DIR / "edge_weight_hist.png"

ICE_TAG = "us-news/ice-us-immigration-and-customs-enforcement"
# stessa soglia del grafo diluito NYT (notebook 02)
MIN_EDGE_WEIGHT = 5

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", force=True)
log = logging.getLogger("guardian_topic_network")

PLOTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"BASE_DIR: {BASE_DIR}")

## 1. Caricamento articoli e tag

Il Parquet conserva solo gli id dei tag (`keyword_tags_json`). Nome leggibile (`webTitle`) e tipo (`keywordType`) stanno nel raw: li rileggiamo da lì, senza richiamare l'API.

Il `keywordType` del Guardian è rimappato sulle categorie del NYT, così i due grafi si colorano allo stesso modo in Gephi. `event` non ha un equivalente nel NYT (che mette gli eventi tra i `subject`) e finisce in `subject`, come `other`.

Il tag ICE è escluso dai nodi per lo stesso motivo del NYT: sarebbe un hub che non dice nulla. Attenzione però: nel NYT la keyword ICE è in tutti gli articoli per costruzione, nel Guardian il tag è in poco più di metà del corpus, perché è applicato quasi solo dal 2025.

In [ ]:
CATEGORY_MAP = {
    "other": "subject",
    "event": "subject",
    "person": "persons",
    "place": "glocations",
    "organisation": "organizations",
    "work-of-art-or-product": "creative_works",
}


def load_tag_metadata() -> dict[str, dict]:
    """Dal raw: {tag_id: {"label": webTitle, "category": categoria NYT equivalente}}."""
    tags = {}
    for path in sorted(RAW_DIR.glob("*/*.json")):
        for page in json.loads(path.read_text()):
            for result in page["results"]:
                for tag in result.get("tags") or []:
                    tags[tag["id"]] = {
                        "label": tag["webTitle"],
                        "category": CATEGORY_MAP.get(tag.get("keywordType"), "subject"),
                    }
    return tags


def load_ice_articles() -> pd.DataFrame:
    """Articoli Guardian rilevanti (colonne: id, pub_date, keywords = lista di tag id)."""
    df = pd.read_parquet(IN_RELEVANT, columns=["id", "pub_date", "keyword_tags_json", "is_ice_relevant"])
    df = df[df["is_ice_relevant"]].copy()
    df["keywords"] = df["keyword_tags_json"].apply(json.loads)
    return df[["id", "pub_date", "keywords"]]


def extract_keywords(keywords: list) -> list[tuple[str, int, str]]:
    """Da lista di tag id a lista (tag_id, rank, category) di UN SOLO articolo, senza il tag ICE."""
    keywords_filtered = []
    seen = set()
    for rank, tag_id in enumerate(keywords, start=1):
        if tag_id == ICE_TAG or tag_id in seen:
            continue
        seen.add(tag_id)
        keywords_filtered.append((tag_id, rank, TAG_META.get(tag_id, {}).get("category", "subject")))
    return keywords_filtered


TAG_META = load_tag_metadata()
df = load_ice_articles()
n_tags = df["keywords"].apply(lambda tags: len(extract_keywords(tags)))
log.info("Articoli Guardian rilevanti: %d, tag noti dal raw: %d", len(df), len(TAG_META))
log.info("Tag per articolo (senza ICE): mediana %d, media %.1f, articoli senza tag %d", n_tags.median(), n_tags.mean(), (n_tags == 0).sum())

In [ ]:
# Demo: tag estratti dal primo articolo
sample_row = df.iloc[0]
print(f"Articolo: {sample_row['id']}")
extract_keywords(sample_row["keywords"])

## 2. Grafo di co-occorrenza (corpus intero)

Stessa funzione del notebook 02. Unica aggiunta: l'attributo `label` (nome leggibile del tag), che Gephi usa come etichetta del nodo al posto dell'id.

In [ ]:
def build_cooccurrence_graph(df: pd.DataFrame) -> nx.Graph:
    """Costruisce il grafo di co-occorrenza: un arco per ogni coppia di keyword nello stesso articolo."""
    graph = nx.Graph()

    for _, row in df.iterrows():
        keywords = extract_keywords(row["keywords"])

        for val, rank, category in keywords:
            if graph.has_node(val):
                graph.nodes[val]["count"] += 1
                graph.nodes[val]["sum_rank"] += rank
                graph.nodes[val]["category_counts"][category] = (
                    graph.nodes[val]["category_counts"].get(category, 0) + 1
                )
            else:
                graph.add_node(val, count=1, sum_rank=rank, category_counts={category: 1})

        if len(keywords) < 2:
            continue

        keywords_values = [kw[0] for kw in keywords]
        edges = combinations(keywords_values, 2)

        for u, v in edges:
            if graph.has_edge(u, v):
                graph[u][v]["weight"] += 1
            else:
                graph.add_edge(u, v, weight=1)

    for node, data in graph.nodes(data=True):
        data["avg_rank"] = data["sum_rank"] / data["count"] if data["count"] > 0 else 0
        category_counts = data.pop("category_counts")
        data["category"] = max(category_counts, key=category_counts.get)
        data["label"] = TAG_META.get(node, {}).get("label", node)

    return graph


def export_graph(graph: nx.Graph, path: Path) -> None:
    """Scrive il grafo in formato GEXF (path)."""
    nx.write_gexf(graph, path)


log.info("Costruisco co-occurrence network su %d articoli (corpus intero)", len(df))
graph = build_cooccurrence_graph(df)
log.info("Grafo intero: %d nodi, %d archi", graph.number_of_nodes(), graph.number_of_edges())
export_graph(graph, OUT_GEXF)
log.info("Scritto %s", OUT_GEXF.relative_to(BASE_DIR))

In [ ]:
# Sanity check: i nodi piu' frequenti devono essere temi coerenti (nessun rumore)
top_nodes = sorted(graph.nodes(data=True), key=lambda x: (-x[1]["count"], x[0]))[:15]
pd.DataFrame(
    [{"tag": n, "label": d["label"], "count": d["count"], "avg_rank": round(d["avg_rank"], 2), "category": d["category"]} for n, d in top_nodes]
)

## 3. Grafo diluito: soglia sui pesi degli archi

Stessa procedura e stessa soglia del NYT (`MIN_EDGE_WEIGHT = 5`): istogramma dei pesi, tabella di cosa resta a ogni soglia, poi grafo con i soli archi di peso ≥ soglia e senza nodi isolati.

In [ ]:
def prune_graph(graph: nx.Graph, min_weight: int) -> nx.Graph:
    """Copia del grafo con i soli archi di peso >= min_weight; i nodi rimasti isolati sono rimossi."""
    pruned = graph.copy()
    pruned.remove_edges_from([(u, v) for u, v, w in graph.edges(data="weight") if w < min_weight])
    pruned.remove_nodes_from(list(nx.isolates(pruned)))
    return pruned


def threshold_table(graph: nx.Graph, thresholds: list[int]) -> pd.DataFrame:
    """Per ogni soglia: nodi/archi rimasti, componente gigante e quota del peso totale conservata."""
    total_weight = sum(w for _, _, w in graph.edges(data="weight"))
    rows = []
    for t in thresholds:
        pruned = prune_graph(graph, t)
        giant = max((len(c) for c in nx.connected_components(pruned)), default=0)
        rows.append({
            "soglia": t,
            "nodi": pruned.number_of_nodes(),
            "archi": pruned.number_of_edges(),
            "componenti": nx.number_connected_components(pruned),
            "nodi_componente_gigante": giant,
            "peso_conservato_pct": round(sum(w for _, _, w in pruned.edges(data="weight")) / total_weight * 100, 1),
        })
    return pd.DataFrame(rows)


def plot_weight_histogram(graph: nx.Graph, min_weight: int, out_path: Path) -> None:
    weights = np.array([w for _, _, w in graph.edges(data="weight")])
    # bin logaritmici: pochi archi molto pesanti e una maggioranza con peso 1
    bins = np.unique(np.round(np.logspace(0, np.log10(weights.max() + 1), 30))) - 0.5

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.hist(weights, bins=bins, color="#1f77b4", edgecolor="white", linewidth=0.5)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.axvline(min_weight - 0.5, color="#d62728", linestyle="--", linewidth=1.5, label=f"soglia = {min_weight}")
    ax.set_xlabel("Peso dell'arco (articoli in cui i due tag compaiono insieme)")
    ax.set_ylabel("Numero di archi")
    ax.set_title("Guardian — Distribuzione dei pesi degli archi nel grafo di co-occorrenza ICE")
    ax.legend()
    ax.grid(alpha=0.3, which="both")
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


plot_weight_histogram(graph, MIN_EDGE_WEIGHT, OUT_WEIGHT_HIST_PNG)
log.info("Scritto %s", OUT_WEIGHT_HIST_PNG.relative_to(BASE_DIR))
display(Image(filename=str(OUT_WEIGHT_HIST_PNG)))
threshold_table(graph, [1, 2, 3, 4, 5, 8, 10, 15, 20, 30, 50])

In [ ]:
pruned_graph = prune_graph(graph, MIN_EDGE_WEIGHT)
log.info(
    "Grafo diluito (peso >= %d): %d nodi, %d archi (da %d nodi, %d archi)",
    MIN_EDGE_WEIGHT, pruned_graph.number_of_nodes(), pruned_graph.number_of_edges(),
    graph.number_of_nodes(), graph.number_of_edges(),
)
export_graph(pruned_graph, OUT_PRUNED_GEXF)
log.info("Scritto %s", OUT_PRUNED_GEXF.relative_to(BASE_DIR))

# Archi piu' pesanti: devono essere coppie di temi sensate
top_edges = sorted(pruned_graph.edges(data="weight"), key=lambda e: (-e[2], e[0], e[1]))[:15]
pd.DataFrame(
    [(graph.nodes[u]["label"], graph.nodes[v]["label"], w) for u, v, w in top_edges],
    columns=["tag_a", "tag_b", "weight"],
)

## 4. Grafi per periodo e grafo dinamico

Stessi quattro periodi del NYT, stesse metriche, stesso formato del grafo dinamico per la Timeline di Gephi.

In [ ]:
def get_periods(df: pd.DataFrame) -> list[tuple[str, int, int]]:
    """Confini dei periodi da confrontare: (label, start_year incluso, end_year escluso)."""
    max_year = int(df["pub_date"].dt.year.max())
    return [
        ("2003-2016", 2003, 2017),
        ("2017-2020", 2017, 2021),
        ("2021-2024", 2021, 2025),
        ("2025-presente", 2025, max_year + 1),
    ]


def filter_by_period(df: pd.DataFrame, start_year: int, end_year: int) -> pd.DataFrame:
    """Filtra df per anno di pubblicazione in [start_year, end_year)."""
    years = df["pub_date"].dt.year
    return df[(years >= start_year) & (years < end_year)]


def graph_stats(graph: nx.Graph) -> dict:
    """Metriche di confronto rapido tra grafi diversi."""
    n_nodes = graph.number_of_nodes()
    n_edges = graph.number_of_edges()
    density = nx.density(graph) if n_nodes > 1 else 0.0
    avg_weighted_degree = (
        sum(w for _, _, w in graph.edges(data="weight")) * 2 / n_nodes if n_nodes else 0.0
    )
    return {"nodes": n_nodes, "edges": n_edges, "density": density, "avg_weighted_degree": avg_weighted_degree}


def build_period_graphs(df: pd.DataFrame, periods: list[tuple[str, int, int]]) -> dict:
    """Costruisce un grafo di co-occorrenza per ciascun periodo. Ritorna {label: (graph, start, end, n_articoli)}."""
    period_graphs = {}
    for label, start, end in periods:
        period_df = filter_by_period(df, start, end)
        pgraph = build_cooccurrence_graph(period_df)
        period_graphs[label] = (pgraph, start, end, len(period_df))
        log.info(
            "Periodo %s (%d articoli): %d nodi, %d archi",
            label, len(period_df), pgraph.number_of_nodes(), pgraph.number_of_edges(),
        )
    return period_graphs


def export_period_stats(period_graphs: dict, path: Path) -> None:
    """CSV di confronto (nodi/archi/densita'/grado pesato medio) tra periodi."""
    rows = []
    for label, (pgraph, start, end, n_articles) in period_graphs.items():
        rows.append({
            "period": label, "start_year": start, "end_year_excl": end,
            "n_articles": n_articles, **graph_stats(pgraph),
        })
    pd.DataFrame(rows).to_csv(path, index=False)


def build_dynamic_graph(period_graphs: dict) -> nx.Graph:
    """Unisce i grafi per periodo in un unico grafo dinamico per la Timeline di Gephi."""
    merged = nx.Graph()
    merged.graph["mode"] = "dynamic"

    for label, (pgraph, start, end, _n_articles) in period_graphs.items():
        start, end = float(start), float(end)
        for node, data in pgraph.nodes(data=True):
            if not merged.has_node(node):
                merged.add_node(node, label=data["label"], count=0, spells=[], count_by_period=[])
            merged.nodes[node]["count"] += data["count"]
            merged.nodes[node]["spells"].append((start, end))
            merged.nodes[node]["count_by_period"].append((data["count"], start, end))

        for u, v, edata in pgraph.edges(data=True):
            if not merged.has_edge(u, v):
                merged.add_edge(u, v, weight=0, spells=[], weight_by_period=[])
            merged[u][v]["weight"] += edata["weight"]
            merged[u][v]["spells"].append((start, end))
            merged[u][v]["weight_by_period"].append((edata["weight"], start, end))

    return merged


periods = get_periods(df)
period_graphs = build_period_graphs(df, periods)

export_period_stats(period_graphs, OUT_PERIOD_STATS)
log.info("Scritto %s", OUT_PERIOD_STATS.relative_to(BASE_DIR))

for label, (period_graph, *_rest) in period_graphs.items():
    out_path = PROCESSED_DIR / f"ice_topic_network_{label}.gexf"
    export_graph(period_graph, out_path)
    log.info("Scritto %s", out_path.relative_to(BASE_DIR))

dynamic_graph = build_dynamic_graph(period_graphs)
export_graph(dynamic_graph, OUT_DYNAMIC_GEXF)
log.info(
    "Scritto %s (%d nodi, %d archi, mode=dynamic)",
    OUT_DYNAMIC_GEXF.relative_to(BASE_DIR), dynamic_graph.number_of_nodes(), dynamic_graph.number_of_edges(),
)

pd.read_csv(OUT_PERIOD_STATS)

## 5. Confronto di struttura NYT vs Guardian

Prime metriche affiancate, sui grafi interi e su quelli diluiti. I grafi NYT sono letti dai GEXF scritti dal notebook 02 (va eseguito prima).

Le due tassonomie non sono allineate: i nodi hanno nomi diversi (`illegal immigration` nel NYT, `US immigration` nel Guardian) e il NYT assegna più keyword per articolo. Qui si confronta la **forma** dei grafi, non i singoli nodi.

In [ ]:
def comparison_row(name: str, graph: nx.Graph) -> dict:
    giant = max((len(c) for c in nx.connected_components(graph)), default=0)
    weights = np.array([w for _, _, w in graph.edges(data="weight")])
    return {
        "grafo": name,
        **graph_stats(graph),
        "componenti": nx.number_connected_components(graph),
        "nodi_componente_gigante": giant,
        "peso_mediano": float(np.median(weights)),
        "peso_max": int(weights.max()),
        "archi_peso_1_pct": round(float((weights == 1).mean() * 100), 1),
        "clustering_medio": round(nx.average_clustering(graph), 3),
    }


comparison = pd.DataFrame([
    comparison_row("NYT intero", nx.read_gexf(IN_NYT_GEXF)),
    comparison_row("Guardian intero", graph),
    comparison_row(f"NYT diluito (>= {MIN_EDGE_WEIGHT})", nx.read_gexf(IN_NYT_PRUNED_GEXF)),
    comparison_row(f"Guardian diluito (>= {MIN_EDGE_WEIGHT})", pruned_graph),
])
comparison.to_csv(OUT_COMPARISON, index=False)
log.info("Scritto %s", OUT_COMPARISON.relative_to(BASE_DIR))
comparison